In [31]:
pip install torch

In [32]:
import torch

def scaled_dot_product_attention(query, key, value, mask=None):
  Q = query
  K = key
  V = value
  d_k = Q.shape[-1]

  # print("Q.shape", Q.shape)
  # print("K.shape", K.shape)
  # print("V.shape", V.shape)
  # print("d_k", d_k)


  # Dot product
  dot_product = torch.matmul(Q, K.transpose(-2,-1))
  # print(dot_product)
  # print(dot_product.shape)

  # Scale
  scaled_dot_product = dot_product / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
  # print(scaled_dot_product)
  # print(scaled_dot_product.shape)

  #Masking for decoder
  if mask is not None:
    scaled_dot_product = scaled_dot_product.masked_fill(mask == 0, float("-inf"))
  # print(scaled_dot_product)
  # print(scaled_dot_product.shape
  #Softmax
  softMaxOperation = torch.nn.Softmax(dim=-1)
  attention_weights = softMaxOperation(scaled_dot_product)
  # print(attention_weights)
  # print(attention_weights.shape)

  # for i in attention_weights:
  #   sum = 0
  #   for j in i :
  #     sum += j
  #   print(sum)

  # #Weighted sum
  weighted_sum = torch.matmul(attention_weights, V)
  # print(weighted_sum)

  return weighted_sum, attention_weights





In [33]:
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)


    def forward(self,q,k,v,mask=None):
      batch_size = q.shape[0]
      q_n = q.shape[1]
      k_n = k.shape[1]
      v_n = v.shape[1]

      Q = self.W_Q(q)
      K = self.W_K(k)
      V = self.W_V(v)

      reshaped_q = Q.reshape(batch_size, q_n,self.num_heads, self.d_k)
      reshaped_k = K.reshape(batch_size, k_n, self.num_heads, self.d_k)
      reshaped_v = V.reshape(batch_size, v_n,self.num_heads, self.d_k)

      transpose_Q = reshaped_q.transpose(1,2)
      transpose_K = reshaped_k.transpose(1,2)
      transpose_V = reshaped_v.transpose(1,2)

      scaled_dot_product_attention_result, attention_weights   = scaled_dot_product_attention(transpose_Q,transpose_K,transpose_V,mask)
      scaled_dot_product_attention_result  = scaled_dot_product_attention_result.transpose(1,2)
      scaled_dot_product_attention_result  = scaled_dot_product_attention_result.reshape(batch_size, q_n,self.d_model)

      output = self.W_O(scaled_dot_product_attention_result)

      return output

In [34]:
class FeedForward(nn.Module):
  def __init__(self, d_model, d_ff):
    super(FeedForward, self).__init__()

    self.linear1 = nn.Linear(d_model,d_ff)
    self.linear2 = nn.Linear(d_ff,d_model)
    self.relu = nn.ReLU()

  def forward(self,x):
    linear1_output = self.linear1(x)
    relu_output = self.relu(linear1_output)
    linear2_output = self.linear2(relu_output)

    return linear2_output

In [35]:
class add_and_norm(nn.Module):
  def __init__(self, d_model):
    super(add_and_norm, self).__init__()
    self.layer_norm = nn.LayerNorm(d_model)

  def forward(self,x,sublayer_output):
    add_output = x + sublayer_output
    norm_output = self.layer_norm(add_output)

    return norm_output

In [36]:
class encode_layer(nn.Module):
  def __init__(self, d_model, num_heads, d_ff):
    super(encode_layer, self).__init__()
    self.multi_head_attention = MultiHeadAttention(d_model, num_heads)
    self.feed_forward = FeedForward(d_model, d_ff)
    self.add_and_norm1 = add_and_norm(d_model)
    self.add_and_norm2 = add_and_norm(d_model)

  def forward(self,x,mask=None):
    sublayer_output = self.multi_head_attention(x,x,x,mask)
    add_and_norm1_output = self.add_and_norm1(x,sublayer_output)
    sublayer_output2 = self.feed_forward(add_and_norm1_output)
    add_and_norm2_output = self.add_and_norm2(add_and_norm1_output,sublayer_output2)

    return add_and_norm2_output

In [37]:
class decode_layer(nn.Module):
  def __init__(self, d_model, num_heads, d_ff):
    super(decode_layer, self).__init__()
    self.masked_multi_head_attention = MultiHeadAttention(d_model, num_heads)
    self.multi_head_attention = MultiHeadAttention(d_model, num_heads)
    self.feed_forward = FeedForward(d_model, d_ff)
    self.add_and_norm1 = add_and_norm(d_model)
    self.add_and_norm2 = add_and_norm(d_model)
    self.add_and_norm3 = add_and_norm(d_model)

  def forward(self,x,encoder_output,mask=None):
    sublayer_output = self.masked_multi_head_attention(x,x,x,mask)
    add_and_norm1_output = self.add_and_norm1(x,sublayer_output)
    sublayer_output2 = self.multi_head_attention(add_and_norm1_output,encoder_output,encoder_output,mask=None)
    add_and_norm2_output = self.add_and_norm2(add_and_norm1_output,sublayer_output2)
    sublayer_output3 = self.feed_forward(add_and_norm2_output)
    add_and_norm3_output = self.add_and_norm3(add_and_norm2_output,sublayer_output3)

    return add_and_norm3_output

In [38]:
class Encoder(nn.Module):
  def __init__(self, num_layers, d_model, num_heads, d_ff):
    super(Encoder, self).__init__()
    self.layers = nn.ModuleList([encode_layer(d_model, num_heads, d_ff) for _ in range(num_layers)])

  def forward(self,x,mask=None):
    for layer in self.layers:
      x = layer(x,mask)

    return x

In [39]:
class Decoder(nn.Module):
  def __init__(self, num_layers, d_model, num_heads, d_ff):
    super(Decoder, self).__init__()
    self.layers = nn.ModuleList([decode_layer(d_model, num_heads, d_ff) for _ in range(num_layers)])

  def forward(self,x,encoder_output,mask=None):
    for layer in self.layers:
      x = layer(x,encoder_output,mask)

    return x

In [40]:
class PostionalEncoding(nn.Module):
  def __init__(self, d_model, max_seq_length):
    super(PostionalEncoding, self).__init__()

    pe = torch.zeros(max_seq_length, d_model)

    position = torch.arange(0, max_seq_length).unsqueeze(1)

    div_term = torch.exp(torch.arange(0, d_model, 2) * -(torch.log(torch.tensor(10000.0)) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    self.register_buffer('pe', pe.unsqueeze(0))

  def forward(self, x):
    x = x + self.pe[0, :x.shape[1],:]
    return x

In [41]:
class Transformer(nn.Module):
  def __init__(self, num_layers, d_model, num_heads, d_ff, src_vocab_size, trg_vocab_size, max_seq_length):
    super(Transformer, self).__init__()

    self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
    self.decoder_embedding = nn.Embedding(trg_vocab_size, d_model)
    self.positional_encoding = PostionalEncoding(d_model, max_seq_length)
    self.encoder = Encoder(num_layers, d_model, num_heads, d_ff)
    self.decoder = Decoder(num_layers, d_model, num_heads, d_ff)
    self.fc_out = nn.Linear(d_model, trg_vocab_size)

  def forward(self, src, trg, src_mask, trg_mask):
    src_embedded = self.encoder_embedding(src)
    trg_embedded = self.decoder_embedding(trg)
    src_embedded = self.positional_encoding(src_embedded)
    trg_embedded = self.positional_encoding(trg_embedded)
    encoder_output = self.encoder(src_embedded, src_mask)
    decoder_output = self.decoder(trg_embedded, encoder_output, trg_mask)
    output = self.fc_out(decoder_output)
    return output

In [43]:
import torch.optim as optim

# small transformer for fast training
transformer = Transformer(
    num_layers=3, d_model=256, num_heads=8,
    d_ff=1024, src_vocab_size=50,
    trg_vocab_size=50, max_seq_length=20
)

optimizer = optim.Adam(transformer.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

def generate_batch(batch_size=32, seq_len=10, vocab_size=48):
    src = torch.randint(3, vocab_size, (batch_size, seq_len))
    trg_input = torch.cat([torch.ones(batch_size, 1).long(), src], dim=1)   # prepend <START>=1
    trg_output = torch.cat([src, torch.full((batch_size, 1), 2).long()], dim=1)  # append <END>=2
    return src, trg_input, trg_output

def create_mask(seq_len):
    return torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)

# training loop
for step in range(1000):
    transformer.train()

    src, trg_input, trg_output = generate_batch()
    trg_mask = create_mask(trg_input.shape[1])

    # forward pass
    output = transformer(src, trg_input,src_mask=None, trg_mask=trg_mask)
    # output shape: (batch, seq_len, vocab_size)

    # reshape for loss:
    # criterion expects (batch*seq_len, vocab_size) and (batch*seq_len,)
    loss = criterion(
        output.reshape(-1, 50),      # (batch*seq_len, vocab_size)
        trg_output.reshape(-1)       # (batch*seq_len,)
    )

    # backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step {step}, loss: {loss.item():.4f}")

step 0, loss: 4.0962
step 100, loss: 2.1066
step 200, loss: 0.0460
step 300, loss: 0.0141
step 400, loss: 0.0085
step 500, loss: 0.0060
step 600, loss: 0.0044
step 700, loss: 0.0037
step 800, loss: 0.0030
step 900, loss: 0.0026


In [44]:
transformer.eval()
src = torch.tensor([[5, 12, 7, 3, 19, 8, 14, 2, 11, 6]])  # one sequence
trg_input = torch.tensor([[1]])  # just START token

# generate one token at a time
for _ in range(10):
    trg_mask = create_mask(trg_input.shape[1])
    output = transformer(src, trg_input, src_mask = None,trg_mask=trg_mask)
    next_token = output[:, -1, :].argmax(dim=-1, keepdim=True)
    trg_input = torch.cat([trg_input, next_token], dim=1)

print("Input:    ", src[0].tolist())
print("Predicted:", trg_input[0, 1:].tolist())  # skip START token

Input:     [5, 12, 7, 3, 19, 8, 14, 2, 11, 6]
Predicted: [5, 12, 7, 3, 19, 8, 14, 28, 11, 6]
